# Lab 1: Keras Neural Network with W&B Experiment Tracking

This notebook trains a Keras neural network on the **Breast Cancer Wisconsin** dataset and logs metrics, predictions, and confusion matrix to **Weights & Biases (W&B)**.

**Changes from the original lab (XGBoost on Dermatology):**
- Replaced XGBoost with a Keras Sequential neural network
- Used Breast Cancer dataset (sklearn built-in) instead of Dermatology
- Used `wandb.keras.WandbCallback()` instead of `wandb.xgboost.WandbCallback()`
- Replaced `DMatrix` with NumPy arrays
- Replaced `bst.predict()` with `model.predict()` + argmax

In [1]:
import os
import wandb
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

wandb.login()

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


wandb: Currently logged in as: krovvidipranav3 (krovvidipranav3-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
# Load the Breast Cancer dataset
data = load_breast_cancer()
X = data.data
y = data.target

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Classes: {data.target_names}")

Train shape: (398, 30), Test shape: (171, 30)
Number of features: 30
Classes: ['malignant' 'benign']


In [3]:
# Initialize W&B run
run = wandb.init(project="Lab1-visualize-models", name="keras_breast_cancer")

# Log config
config = {
    "model_type": "keras_sequential",
    "dataset": "breast_cancer",
    "epochs": 20,
    "batch_size": 32,
    "learning_rate": 0.001,
    "hidden_units": [64, 32],
    "dropout": 0.3,
}
wandb.config.update(config)

wandb: setting up run hes3iv1l


wandb: Tracking run with wandb version 0.25.1


wandb: Run data is saved locally in /Users/saipranavkrovvidi/Documents/Masters/Masters Sem 3-Spring 2026/MLOps/Labs/MLOps_Labs/experiment_tracking_lab6/wandb/run-20260403_215048-hes3iv1l
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run keras_breast_cancer


wandb: ⭐️ View project at https://wandb.ai/krovvidipranav3-northeastern-university/Lab1-visualize-models


wandb: 🚀 View run at https://wandb.ai/krovvidipranav3-northeastern-university/Lab1-visualize-models/runs/hes3iv1l


In [4]:
# Build Keras model
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(2, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,130 (16.13 KB)

 Trainable params: 4,130 (16.13 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Train with W&B Keras callback
# Disabled model saving features to avoid Keras 3 compatibility issues
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=config["epochs"],
    batch_size=config["batch_size"],
    callbacks=[
        wandb.keras.WandbCallback(
            log_model=False,
            save_model=False,
            save_graph=False
        )
    ],
    verbose=1
)

wandb: WARNING WandbCallback is deprecated and will be removed in a future release. Please use the WandbMetricsLogger, WandbModelCheckpoint, and WandbEvalCallback callbacks instead. See https://docs.wandb.ai/guides/integrations/keras for more information.


Epoch 1/20


 1/13 ━━━━━━━━━━━━━━━━━━━━ 9s 830ms/step - accuracy: 0.4062 - loss: 0.8842

13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5704 - loss: 0.7784 - val_accuracy: 0.8713 - val_loss: 0.4429


Epoch 2/20


...

 1/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.0294

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9799 - loss: 0.0753 - val_accuracy: 0.9766 - val_loss: 0.0662


Epoch 20/20


 1/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.0280

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9698 - loss: 0.0741 - val_accuracy: 0.9766 - val_loss: 0.0657


In [6]:
# Get predictions
pred_probs = model.predict(X_test)
pred = np.argmax(pred_probs, axis=1)

error_rate = np.sum(pred != y_test) / y_test.shape[0]
accuracy = 1 - error_rate
print(f"Test accuracy = {accuracy:.4f}")
print(f"Test error rate = {error_rate:.4f}")

run.summary['Error Rate'] = error_rate
run.summary['Accuracy'] = accuracy

1/6 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


Test accuracy = 0.9766
Test error rate = 0.0234


In [7]:
# Log confusion matrix to W&B
class_names = list(data.target_names)
cm = wandb.plot.confusion_matrix(
    probs=None,
    y_true=y_test.tolist(),
    preds=pred.tolist(),
    class_names=class_names
)
wandb.log({"confusion_matrix": cm})

In [8]:
run.finish()

wandb: updating run metadata; uploading artifact run-hes3iv1l-confusion_matrix_table


wandb: uploading artifact run-hes3iv1l-confusion_matrix_table


wandb: uploading history steps 0-20, summary, console lines 0-45


wandb: 
wandb: Run history:
wandb:     accuracy ▁▅▇▇▇▇▇▇▇███████████
wandb:        epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
wandb:         loss █▅▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
wandb: val_accuracy ▁▅▆▆▆▆▇▇██████▇▇████
wandb:     val_loss █▅▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:      Accuracy 0.97661
wandb:    Error Rate 0.02339
wandb:      accuracy 0.96985
wandb:    best_epoch 17
wandb: best_val_loss 0.06506
wandb:         epoch 19
wandb:          loss 0.07409
wandb:  val_accuracy 0.97661
wandb:      val_loss 0.06571
wandb: 


wandb: 🚀 View run keras_breast_cancer at: https://wandb.ai/krovvidipranav3-northeastern-university/Lab1-visualize-models/runs/hes3iv1l
wandb: ⭐️ View project at: https://wandb.ai/krovvidipranav3-northeastern-university/Lab1-visualize-models
wandb: Synced 5 W&B file(s), 1 media file(s), 2 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260403_215048-hes3iv1l/logs
